In [ ]:
import sys
sys.path.append("..")
import torch

from src.environment import MultiCurrencyEnv
from src.simulator import geometric_brownian_step
from src.data import load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
from bokeh.models import HoverTool, ColumnDataSource
import bokeh.plotting as bk
bk.output_notebook()

# Simulate Data

In [ ]:
N = 4
C0 = 1_000.0
w0 = torch.zeros(N)
p0 = torch.tensor([100., 50., 25., 10.])
sell_fee = torch.full((N,), 0.001)
buy_fee  = torch.full((N,), 0.001)

taus_price = torch.tensor([1., 5., 10., 20., 50., 100.])
vol_taus   = torch.tensor([10., 50., 100.])
volu_taus  = torch.tensor([10., 50., 100.])

env = MultiCurrencyEnv(C0, w0, p0, sell_fee, buy_fee,
                       taus_price, dt=1.0,
                       vol_taus=vol_taus, volu_taus=volu_taus,
                       reward_mode="log",transaction_eps=1e-4)

a_hist = []
V_hist = []
C_hist = []
w_hist = []
prices_hist = []
volume_hist = []

p_rel_hist = []
x_hist = []
sigma_hist = []
v_rel_hist = []
rew_hist = []

T = 1000

state = env.reset()
for t in range(T):
    a = torch.tanh(torch.randn(N+1))

    new_prices = geometric_brownian_step(env.p, mu=0.001, sigma=0.05)
    new_volume = torch.rand(N) * 1000
    env.queue_update({
        'time': 0,
        'prices': new_prices,
        'volume': new_volume
    })
    state, reward, done, info = env.step(a)

    prices_hist.append(new_prices)
    volume_hist.append(new_volume)

    a_hist.append(info['a'])
    V_hist.append(info['V'])
    C_hist.append(info['C'])
    w_hist.append(info['w'])

    p_rel_hist.append(state.p_rel.detach().cpu().numpy())
    x_hist.append(state.x.detach().cpu().numpy())
    sigma_hist.append(state.sigma.detach().cpu().numpy())
    v_rel_hist.append(state.v_rel.detach().cpu().numpy())
    rew_hist.append(reward)


In [ ]:
t = list(range(T))

In [ ]:
f1 = bk.figure(title=f"Prices", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    pi = [prices_hist[k][i] for k in range(T)]
    r = f1.line(t, pi, line_width=2, legend_label=f"Asset {i+1}", color=Category10[10][i])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Volume", x_axis_label="t", y_axis_label="USD", width=900, height=320)

for i in range(N):
    vi = [volume_hist[k][i] * prices_hist[k][i] for k in range(T)]
    r = f1.line(t, vi, line_width=2, legend_label=f"Asset {i+1}", color=Category10[10][i])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Portfolio Value", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
r1 = f1.line(t, V_hist, line_width=2, legend_label="Total V")
r2 = f1.line(t, C_hist, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
f1.add_tools(HoverTool(renderers=[r1], tooltips=[("t", "@t"), ("V", "@V{0,0.00}")], mode="vline"))
bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Rewards", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

r = f1.line(t, rew_hist, line_width=2, legend_label=f"Reward")

ret_hist = []
gamma = 0.99
G = 0.0
for r in reversed(rew_hist):
    G = r + gamma * G
    ret_hist.insert(0, G)

r = f1.line(t, ret_hist, line_width=2, legend_label=f"Return", color="green")
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Actions", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

a0 = [a_hist[k][0] for k in range(T)]
f1.scatter(t, a0, legend_label=f"Investment fraction", color=Category10[10][0])
for i in range(1,N+1):
    ai = [a_hist[k][i] for k in range(T)]
    f1.scatter(t, ai, legend_label=f"Asset {i} fraction to buy/sell", color=Category10[10][i])

f1.legend.click_policy = "hide"
bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Asset Position", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

x0 = [x_hist[k][0] for k in range(T)]
f1.line(t, x0, line_width=2, legend_label=f"Cash fraction", color=Category10[10][0])
for i in range(1,N+1):
    xi = [x_hist[k][i] for k in range(T)]
    f1.line(t, xi, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Relative price change (EWMA)", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    for j in range(len(env.taus_p)):
        p_ij = [p_rel_hist[k][j,i] for k in range(T)]
        f1.line(t, p_ij, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i], alpha=0.2 + 0.8 * j / len(env.taus_p))
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Relative Volatility (EWMA)", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    for j in range(len(env.vol_taus)):
        sigma_ij = [sigma_hist[k][j,i] for k in range(T)]
        f1.line(t, sigma_ij, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i], alpha=0.2 + 0.8 * j / len(env.vol_taus))
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Relative Volume (EWMA)", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    for j in range(len(env.volu_taus)):
        v_rel_ij = [v_rel_hist[k][j,i] for k in range(T)]
        f1.line(t, v_rel_ij, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i], alpha=0.2 + 0.8 * j / len(env.volu_taus))
f1.legend.click_policy = "hide"

bk.show(f1)